# Task 055: batman — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Exoplanet transit light curve fitting using batman

**Paper**: batman: BAsic Transit Model cAlculatioN in Python
**Repository**: https://github.com/lkreidberg/batman

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 62.77 dB
- **SSIM**: N/A (1D light curve)
- **Evaluation**: Exoplanet transit light curve fitting (correlation_flux=0.9999995, RMSE=12.3 ppm)

### Mathematical Formulation

**Parameter estimation** fits model parameters $\boldsymbol{\theta}$ to data:

$$\hat{\boldsymbol{\theta}} = \arg\min_{\boldsymbol{\theta}} \sum_{i=1}^{N} \left(\frac{y_i - f(t_i; \boldsymbol{\theta})}{\sigma_i}\right)^2$$

**Fisher information matrix**: $\mathcal{I}_{jk} = \sum_i \frac{1}{\sigma_i^2} \frac{\partial f_i}{\partial\theta_j}\frac{\partial f_i}{\partial\theta_k}$

**Cramer-Rao bound**: $\text{Var}(\hat\theta_j) \geq [\mathcal{I}^{-1}]_{jj}$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
batman — Exoplanet Transit Photometry Inverse Problem
======================================================
Task: Recover planet orbital/physical parameters from a transit light curve.

Inverse Problem:
    Given a time-series light curve F(t) showing a planetary transit dip,
    recover the planet-to-star radius ratio Rp/Rs, orbital inclination i,
    scaled semi-major axis a/Rs, and limb-darkening coefficients.

Forward Model (batman – BAsic Transit Model cAlculatioN):
    F(t) = 1 - δ(t;  Rp/Rs, a/Rs, i, u₁, u₂, period, t0, e, ω)
    Uses Mandel & Agol (2002) analytic formulae implemented in batman.

Inverse Solver:
    scipy.optimize.differential_evolution (global) +
    scipy.optimize.minimize (local Nelder-Mead refinement)

Repo: https://github.com/lkreidberg/batman
Paper: Kreidberg (2015), PASP, 127, 957, 1161–1165.
       doi:10.1086/683602

Usage:
    /data/yjh/spectro_env/bin/python batman_code.py
"""

import numpy as np
import matplotlib

## 3. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `forward_operator`, `load_or_generate_data`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. Forward Operator (batman transit model)
# ═══════════════════════════════════════════════════════════
def forward_operator(params, t):
    """
    Compute transit light curve F(t) using batman.

    batman implements the Mandel & Agol (2002) analytic model
    for planetary transit light curves, supporting:
      - Uniform, linear, quadratic, nonlinear limb darkening
      - Eccentric orbits
      - Secondary eclipses

    Parameters
    ----------
    params : dict  Transit parameters.
    t : np.ndarray Time array [days].

    Returns
    -------
    flux : np.ndarray  Normalised flux F(t).
    """
    bm_params = batman.TransitParams()
    bm_params.rp = params["rp"]
    bm_params.a = params["a"]
    bm_params.inc = params["inc"]
    bm_params.ecc = params["ecc"]
    bm_params.w = params["w"]
    bm_params.t0 = params["t0"]
    bm_params.per = params["per"]
    bm_params.u = [params["u1"], params["u2"]]
    bm_params.limb_dark = "quadratic"

    model = batman.TransitModel(bm_params, t)
    flux = model.light_curve(bm_params)
    return flux

# ═══════════════════════════════════════════════════════════
# 3. Data Generation
# ═══════════════════════════════════════════════════════════
def load_or_generate_data():
    """Generate synthetic transit light curve with batman."""
    print("[DATA] Generating synthetic transit light curve with batman ...")

    t = np.linspace(-T_SPAN, T_SPAN, N_TIME)

    flux_clean = forward_operator(GT_PARAMS, t)

    # Transit depth check
    depth = 1.0 - flux_clean.min()
    print(f"[DATA] Transit depth = {depth*1e6:.0f} ppm  "
          f"(Rp/Rs = {GT_PARAMS['rp']:.3f})")

    # Add Gaussian photometric noise
    rng = np.random.default_rng(SEED)
    sigma = NOISE_PPM * 1e-6  # convert ppm to relative flux
    flux_noisy = flux_clean + sigma * rng.standard_normal(N_TIME)
    flux_err = np.full(N_TIME, sigma)

    print(f"[DATA] Noise = {NOISE_PPM} ppm  |  {N_TIME} points  |  "
          f"T ∈ [{t[0]:.3f}, {t[-1]:.3f}] days")

    return t, flux_noisy, flux_clean, flux_err

## 4. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `reconstruct`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. Inverse Solver
# ═══════════════════════════════════════════════════════════
def reconstruct(t, flux_meas, flux_err):
    """
    Fit transit parameters using DE + Nelder-Mead through batman forward.

    Free parameters: rp, a, inc, u1, u2
    Fixed parameters: t0, per, ecc, w  (assumed known from ephemeris)

    Parameters
    ----------
    t : np.ndarray        Time array.
    flux_meas : np.ndarray Measured (noisy) flux.
    flux_err : np.ndarray  Error bars.

    Returns
    -------
    fit_params : dict  Best-fit parameter values.
    flux_fit : np.ndarray  Best-fit light curve.
    """
    from scipy.optimize import differential_evolution, minimize

    # Fixed parameters
    fixed = {k: GT_PARAMS[k] for k in ["t0", "per", "ecc", "w"]}

    def chi2(x):
        rp, a, inc, u1, u2 = x
        params = {
            "rp": rp, "a": a, "inc": inc,
            "u1": u1, "u2": u2, **fixed
        }
        try:
            model_flux = forward_operator(params, t)
        except Exception:
            return 1e20
        return np.sum(((flux_meas - model_flux) / flux_err) ** 2)

    # Bounds for free parameters
    bounds = [
        (0.01, 0.3),    # rp  (Rp/Rs)
        (2.0, 50.0),    # a   (a/Rs)
        (70.0, 90.0),   # inc [deg]
        (0.0, 0.8),     # u1
        (-0.3, 0.6),    # u2
    ]

    # Stage 1: Differential Evolution
    print("[RECON] Stage 1 — Differential Evolution (global search) ...")
    result_de = differential_evolution(
        chi2, bounds, seed=SEED,
        maxiter=150, tol=1e-5, popsize=15,
        mutation=(0.5, 1.5), recombination=0.8
    )
    print(f"[RECON]   χ² = {result_de.fun:.2f}  "
          f"(reduced χ²/ν = {result_de.fun/N_TIME:.4f})")

    # Stage 2: Nelder-Mead local refinement
    print("[RECON] Stage 2 — Nelder-Mead refinement ...")
    result_nm = minimize(
        chi2, result_de.x, method='Nelder-Mead',
        options={'maxiter': 2000, 'xatol': 1e-6, 'fatol': 1e-6}
    )
    print(f"[RECON]   χ² = {result_nm.fun:.2f}")

    rp, a, inc, u1, u2 = result_nm.x
    fit_params = {
        "rp": float(rp), "a": float(a), "inc": float(inc),
        "u1": float(u1), "u2": float(u2), **fixed
    }

    flux_fit = forward_operator(fit_params, t)
    return fit_params, flux_fit

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize_results`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. Metrics
# ═══════════════════════════════════════════════════════════
def compute_metrics(gt_params, fit_params, flux_clean, flux_fit, t):
    """
    Compute light-curve and parameter-recovery metrics.
    """
    from skimage.metrics import structural_similarity as ssim_fn

    # Light-curve metrics
    residual = flux_clean - flux_fit
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    cc = float(np.corrcoef(flux_clean, flux_fit)[0, 1])

    data_range = flux_clean.max() - flux_clean.min()
    mse = np.mean(residual ** 2)
    psnr = float(10 * np.log10(data_range ** 2 / max(mse, 1e-30)))

    tile_rows = 7
    a2d = np.tile(flux_clean, (tile_rows, 1))
    b2d = np.tile(flux_fit, (tile_rows, 1))
    ssim = float(ssim_fn(
        a2d, b2d,
        data_range=data_range, win_size=7
    ))

    # Relative error
    norm_gt = np.linalg.norm(flux_clean)
    re = float(np.linalg.norm(residual) / max(norm_gt, 1e-12))

    # Parameter recovery
    free_keys = ["rp", "a", "inc", "u1", "u2"]
    param_metrics = {}
    for k in free_keys:
        gt_v = gt_params[k]
        fit_v = fit_params[k]
        param_metrics[f"gt_{k}"] = float(gt_v)
        param_metrics[f"fit_{k}"] = float(fit_v)
        param_metrics[f"abs_err_{k}"] = float(abs(gt_v - fit_v))
        if abs(gt_v) > 1e-12:
            param_metrics[f"rel_err_{k}_pct"] = float(abs(gt_v - fit_v) / abs(gt_v) * 100)

    metrics = {
        "PSNR": psnr,
        "SSIM": ssim,
        "CC": cc,
        "RMSE": rmse,
        "RE": re,
        **param_metrics,
    }
    return metrics

# ═══════════════════════════════════════════════════════════
# 6. Visualization
# ═══════════════════════════════════════════════════════════
def visualize_results(t, flux_meas, flux_clean, flux_fit,
                      gt_params, fit_params, metrics, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # (a) Light curves
    ax = axes[0, 0]
    ax.plot(t * 24, flux_meas, 'k.', ms=1, alpha=0.3, label='Noisy data')
    ax.plot(t * 24, flux_clean, 'b-', lw=2, label='Ground truth')
    ax.plot(t * 24, flux_fit, 'r--', lw=1.5, label='batman fit')
    ax.set_xlabel('Time from mid-transit [hours]')
    ax.set_ylabel('Relative flux')
    ax.set_title('(a) Transit Light Curve')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # (b) Residuals
    ax = axes[0, 1]
    residual_ppm = (flux_clean - flux_fit) * 1e6
    ax.plot(t * 24, residual_ppm, 'g-', lw=0.8)
    ax.axhline(0, color='k', ls='--', lw=0.5)
    ax.set_xlabel('Time [hours]')
    ax.set_ylabel('Residual [ppm]')
    ax.set_title(f'(b) Residuals  RMSE={metrics["RMSE"]*1e6:.1f} ppm')
    ax.grid(True, alpha=0.3)

    # (c) Transit depth zoom
    ax = axes[1, 0]
    mask = np.abs(t * 24) < 2  # within ±2 hours
    ax.plot(t[mask] * 24, flux_clean[mask], 'b-', lw=2, label='GT')
    ax.plot(t[mask] * 24, flux_fit[mask], 'r--', lw=2, label='Fit')
    ax.set_xlabel('Time [hours]')
    ax.set_ylabel('Flux')
    ax.set_title('(c) Transit Detail (±2 hr)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # (d) Parameter bar chart
    ax = axes[1, 1]
    keys = ["rp", "a", "inc", "u1", "u2"]
    labels = ["Rp/Rs", "a/Rs", "inc [°]", "u₁", "u₂"]
    gt_vals = [gt_params[k] for k in keys]
    fit_vals = [fit_params[k] for k in keys]
    x = np.arange(len(keys))
    w = 0.35
    ax.bar(x - w/2, gt_vals, w, label='GT', color='steelblue')
    ax.bar(x + w/2, fit_vals, w, label='Fit', color='tomato')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_title('(d) Parameter Recovery')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

    fig.suptitle(
        f"batman — Transit Photometry Inversion\n"
        f"PSNR={metrics['PSNR']:.1f} dB  |  SSIM={metrics['SSIM']:.4f}  |  "
        f"CC={metrics['CC']:.4f}  |  RMSE={metrics['RMSE']*1e6:.1f} ppm",
        fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[VIS] Saved → {save_path}")

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
print("=" * 65)
print("  batman — Exoplanet Transit Photometry Inverse Problem")
print("=" * 65)

t, flux_meas, flux_clean, flux_err = load_or_generate_data()

print("\n[RECON] Fitting transit parameters via batman forward model ...")
fit_params, flux_fit = reconstruct(t, flux_meas, flux_err)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print("\n[EVAL] Computing metrics ...")
metrics = compute_metrics(GT_PARAMS, fit_params, flux_clean, flux_fit, t)
for k, v in sorted(metrics.items()):
    print(f"  {k:30s} = {v}")

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), flux_fit)
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), flux_clean)
np.save(os.path.join(RESULTS_DIR, "measurements.npy"), flux_meas)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
visualize_results(t, flux_meas, flux_clean, flux_fit,
                  GT_PARAMS, fit_params, metrics,
                  os.path.join(RESULTS_DIR, "reconstruction_result.png"))

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("\n" + "=" * 65)
print("  DONE — batman transit photometry benchmark complete")
print("=" * 65)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **batman**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=62.77 dB, SSIM=N/A (1D light curve))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: batman: BAsic Transit Model cAlculatioN in Python
- Repository: https://github.com/lkreidberg/batman
